# Phan tich thi truong chung khoan va du doan gia co phieu su dung PySpark (Time Series)

Notebook nay trinh bay day du quy trinh Data Engineering + Data Science cho bai toan da ma co phieu voi du lieu time series.

## 1. Gioi thieu

### Bai toan
- Muc tieu 1: Phan tich hanh vi thi truong theo tung ma va toan thi truong.
- Muc tieu 2: Du doan gia dong cua ngay tiep theo (regression) va huong tang/giam (classification).

### Vi sao phan tich da ma (multi-stock) quan trong
- Co the quan sat tinh dong bien/phan ky giua cac co phieu.
- Danh gia tuong quan giua thi truong Viet Nam (FPT, HPG, VCB, VIC, VNM) va My (AAPL, TSLA).
- Huu ich cho quan tri rui ro va xay dung danh muc.

### Vi sao dung PySpark va Parquet
- PySpark xu ly du lieu lon, mo rong tot, ho tro Window function manh cho time series.
- Parquet luu tru cot (columnar), doc nhanh, nen tot, giu schema on dinh.

### Nguyen tac ky thuat quan trong
- Du lieu la time series => KHONG shuffle khi tao train/test.
- Moi phep tinh rolling/lag/lead phai partition theo ticker va order theo time.
- Tranh data leakage: feature chi duoc dung thong tin den thoi diem hien tai, khong dung tuong lai.

## 2. Khoi tao moi truong

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import RegressionEvaluator, BinaryClassificationEvaluator, MulticlassClassificationEvaluator

import pandas as pd
import matplotlib.pyplot as plt

spark = (
    SparkSession.builder
    .appName("StockMarket-TimeSeries-PySpark")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Khoi tao SparkSession thanh cong.")

## 3. Load du lieu

Doc du lieu Parquet tu thu muc data/. Neu duong dan nay khong ton tai, notebook se fallback sang cac thu muc Parquet pho bien trong workspace.

In [ ]:
candidate_paths = [
    "data/",
    "data/*.parquet",
    "stocks_data.parquet",
    "data_stocks_features.parquet"
]

df_raw = None
selected_path = None
for p in candidate_paths:
    try:
        tmp = spark.read.parquet(p)
        _ = tmp.limit(1).count()
        df_raw = tmp
        selected_path = p
        break
    except Exception:
        pass

if df_raw is None:
    raise FileNotFoundError("Khong tim thay parquet hop le trong data/. Hay kiem tra duong dan du lieu.")

print("Da load du lieu tu:", selected_path)
df_raw.printSchema()
df_raw.show(10, truncate=False)
print("So dong:", df_raw.count())

## 4. Tien xu ly du lieu

- Chuan hoa ten cot ve dung schema mong doi: time, open, high, low, close, volume, ticker, prev_close, ma7, daily_return.
- Sap xep theo (ticker, time).
- Kiem tra null.

**Tai sao phai giu thu tu thoi gian?**
- Vi rolling/lag/lead phu thuoc truc tiep vao thu tu thoi gian.
- Neu dao tron (shuffle) khi chia train/test, mo hinh se thay thong tin tuong lai trong train va gay leakage.

In [ ]:
# Chuan hoa ten cot
df = df_raw

rename_map = {}
if "date" in df.columns and "time" not in df.columns:
    rename_map["date"] = "time"
if "return" in df.columns and "daily_return" not in df.columns:
    rename_map["return"] = "daily_return"
if "symbol" in df.columns and "ticker" not in df.columns:
    rename_map["symbol"] = "ticker"
if "ma_7" in df.columns and "ma7" not in df.columns:
    rename_map["ma_7"] = "ma7"

for old_col, new_col in rename_map.items():
    df = df.withColumnRenamed(old_col, new_col)

required_cols = ["time", "open", "high", "low", "close", "volume", "ticker"]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Thieu cot bat buoc: {missing_required}")

df = (
    df.withColumn("time", F.to_timestamp("time"))
      .withColumn("open", F.col("open").cast("double"))
      .withColumn("high", F.col("high").cast("double"))
      .withColumn("low", F.col("low").cast("double"))
      .withColumn("close", F.col("close").cast("double"))
      .withColumn("volume", F.col("volume").cast("double"))
      .withColumn("ticker", F.col("ticker").cast("string"))
)

# Neu chua co prev_close/ma7/daily_return thi tao tu du lieu goc (khong leakage)
w_ticker_time = Window.partitionBy("ticker").orderBy("time")
w_ma7 = w_ticker_time.rowsBetween(-6, 0)

if "prev_close" not in df.columns:
    df = df.withColumn("prev_close", F.lag("close", 1).over(w_ticker_time))
if "ma7" not in df.columns:
    df = df.withColumn("ma7", F.avg("close").over(w_ma7))
if "daily_return" not in df.columns:
    df = df.withColumn("daily_return", (F.col("close") - F.col("prev_close")) / F.col("prev_close"))

df = df.orderBy("ticker", "time")

print("Tong so dong sau chuan hoa:", df.count())
print("So ticker:", df.select("ticker").distinct().count())
df.select("ticker", "time", "open", "high", "low", "close", "volume", "prev_close", "ma7", "daily_return").show(10, truncate=False)

In [ ]:
# Kiem tra null
null_exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns]
null_df = df.select(*null_exprs)
null_df.show(truncate=False)

print("Ly do khong shuffle:")
print("- Time series can giu nguyen thu tu de tao lag/rolling dung ban chat qua khu -> hien tai -> tuong lai.")
print("- Shuffle truoc khi split se gay data leakage va overestimate hieu nang model.")

## 5. Phan tich thi truong (EDA nang cao)

### 5.1 Phan tich theo tung ma
Tinh gia trung binh va volatility (std cua daily_return) theo ticker.

In [ ]:
ticker_stats = (
    df.groupBy("ticker")
      .agg(
          F.avg("close").alias("avg_close"),
          F.stddev("daily_return").alias("volatility_return"),
          F.avg("daily_return").alias("avg_return"),
          F.count(F.lit(1)).alias("n_rows")
      )
      .orderBy(F.desc("avg_close"))
)
ticker_stats.show(truncate=False)

### 5.2 Phan tich toan thi truong
Tinh average return theo thoi gian va so sanh nhom VN vs US.

In [ ]:
market_return_by_time = (
    df.groupBy("time")
      .agg(F.avg("daily_return").alias("market_avg_return"))
      .orderBy("time")
)
market_return_by_time.show(10, truncate=False)

us_tickers = ["AAPL", "TSLA"]
vn_tickers = ["FPT", "HPG", "VCB", "VIC", "VNM"]

df_market = df.withColumn(
    "market_group",
    F.when(F.col("ticker").isin(us_tickers), F.lit("US"))
     .when(F.col("ticker").isin(vn_tickers), F.lit("VN"))
     .otherwise(F.lit("OTHER"))
)

group_stats = (
    df_market.groupBy("market_group")
      .agg(
          F.avg("daily_return").alias("avg_return"),
          F.stddev("daily_return").alias("volatility"),
          F.avg("volume").alias("avg_volume")
      )
      .orderBy("market_group")
)
group_stats.show(truncate=False)

### 5.3 Pivot du lieu (quan trong)
Bien doi thanh dang: time -> nhieu cot ticker (gia tri la daily_return) de phuc vu correlation analysis.

In [ ]:
pivot_returns = (
    df.select("time", "ticker", "daily_return")
      .groupBy("time")
      .pivot("ticker")
      .agg(F.first("daily_return"))
      .orderBy("time")
)

pivot_returns.show(10, truncate=False)
print("So cot sau pivot:", len(pivot_returns.columns))

### 5.4 Correlation giua cac co phieu

In [ ]:
tickers = [r[0] for r in df.select("ticker").distinct().orderBy("ticker").collect()]

corr_rows = []
for t1 in tickers:
    row = {"ticker": t1}
    for t2 in tickers:
        if t1 == t2:
            row[t2] = 1.0
        else:
            if t1 in pivot_returns.columns and t2 in pivot_returns.columns:
                row[t2] = pivot_returns.stat.corr(t1, t2)
            else:
                row[t2] = None
    corr_rows.append(row)

corr_pdf = pd.DataFrame(corr_rows).set_index("ticker")
corr_pdf

## 6. Feature Engineering (Time Series)

Su dung Window partition theo ticker, order theo time.
Tao them MA14, MA30, rolling volatility, momentum, lag1, lag3.

In [ ]:
w = Window.partitionBy("ticker").orderBy("time")
w14 = w.rowsBetween(-13, 0)
w30 = w.rowsBetween(-29, 0)

df_feat = (
    df
    .withColumn("MA14", F.avg("close").over(w14))
    .withColumn("MA30", F.avg("close").over(w30))
    .withColumn("rolling_volatility_14", F.stddev("daily_return").over(w14))
    .withColumn("momentum_3", F.col("close") - F.lag("close", 3).over(w))
    .withColumn("lag_close_1", F.lag("close", 1).over(w))
    .withColumn("lag_close_3", F.lag("close", 3).over(w))
    .withColumn("lag_return_1", F.lag("daily_return", 1).over(w))
    .withColumn("lag_return_3", F.lag("daily_return", 3).over(w))
)

df_feat.select(
    "ticker", "time", "close", "ma7", "MA14", "MA30",
    "rolling_volatility_14", "momentum_3", "lag_close_1", "lag_close_3"
).show(20, truncate=False)

## 7. Xay dung bai toan du doan

### 7.1 Tao target
- target_next_close = lead(close, 1) theo tung ticker.
- target_direction = 1 neu target_next_close > close, nguoc lai 0.

### 7.2 Giai thich forecasting
- Time series forecasting du doan gia tri tuong lai dua tren lich su.
- Trong notebook nay, target la gia ngay tiep theo, dam bao khong dung thong tin tuong lai de tao feature.

In [ ]:
df_target = (
    df_feat
    .withColumn("target_next_close", F.lead("close", 1).over(w))
    .withColumn("target_direction", F.when(F.col("target_next_close") > F.col("close"), 1).otherwise(0))
)

df_target.select("ticker", "time", "close", "target_next_close", "target_direction").show(20, truncate=False)

## 8. Chuan bi du lieu cho ML

- Loai bo null do lag/rolling/lead.
- Dung VectorAssembler gom feature.
- Feature chinh: ma7, MA14, volume, daily_return, lag feature, volatility, momentum.

In [ ]:
feature_cols = [
    "ma7", "MA14", "MA30", "volume", "daily_return",
    "rolling_volatility_14", "momentum_3",
    "lag_close_1", "lag_close_3", "lag_return_1", "lag_return_3"
]

ml_df = df_target.dropna(subset=feature_cols + ["target_next_close", "target_direction", "time", "ticker"])

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
ml_df = assembler.transform(ml_df)

print("Tong so dong cho ML:", ml_df.count())
ml_df.select("ticker", "time", "features", "target_next_close", "target_direction").show(5, truncate=True)

## 9. Train model

### 9.1 Chia train/test theo thoi gian (khong random)
Mac dinh split_date = 2024-01-01. Neu dataset khong phu hop moc nay, tu dong fallback chia theo quantile thoi gian 80/20.

### 9.2 Mo hinh
- Linear Regression: du doan gia ngay tiep theo.
- Logistic Regression: du doan tang/giam.

In [ ]:
split_date = F.to_timestamp(F.lit("2024-01-01 00:00:00"))
train_df = ml_df.filter(F.col("time") < split_date)
test_df = ml_df.filter(F.col("time") >= split_date)

train_n = train_df.count()
test_n = test_df.count()

if train_n == 0 or test_n == 0:
    q = ml_df.selectExpr("percentile_approx(unix_timestamp(time), 0.8) as split_ts").collect()[0]["split_ts"]
    train_df = ml_df.filter(F.unix_timestamp("time") < F.lit(q))
    test_df = ml_df.filter(F.unix_timestamp("time") >= F.lit(q))

print("Train rows:", train_df.count())
print("Test rows :", test_df.count())

# Linear Regression
lr_reg = LinearRegression(featuresCol="features", labelCol="target_next_close", maxIter=100, regParam=0.01, elasticNetParam=0.0)
reg_model = lr_reg.fit(train_df)
pred_reg = reg_model.transform(test_df).withColumnRenamed("prediction", "pred_next_close")

# Logistic Regression
lr_clf = LogisticRegression(featuresCol="features", labelCol="target_direction", maxIter=100, regParam=0.01)
clf_model = lr_clf.fit(train_df)
pred_clf = clf_model.transform(test_df).withColumnRenamed("prediction", "pred_direction")

pred_reg.select("ticker", "time", "close", "target_next_close", "pred_next_close").show(10, truncate=False)
pred_clf.select("ticker", "time", "target_direction", "pred_direction", "probability").show(10, truncate=False)

## 10. Danh gia model

- Regression: RMSE
- Classification: Accuracy, AUC

In [ ]:
# Regression metric
reg_evaluator = RegressionEvaluator(labelCol="target_next_close", predictionCol="pred_next_close", metricName="rmse")
rmse = reg_evaluator.evaluate(pred_reg)

# Classification metrics
acc_evaluator = MulticlassClassificationEvaluator(labelCol="target_direction", predictionCol="pred_direction", metricName="accuracy")
accuracy = acc_evaluator.evaluate(pred_clf)

auc_evaluator = BinaryClassificationEvaluator(labelCol="target_direction", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
auc = auc_evaluator.evaluate(pred_clf)

print("RMSE (Regression):", round(rmse, 6))
print("Accuracy (Classification):", round(accuracy, 6))
print("AUC (Classification):", round(auc, 6))

## 11. Visualization

Chi dung Pandas/Matplotlib cho truc quan hoa, khong dung cho xu ly chinh.

In [ ]:
# Chon mot ma de minh hoa
example_ticker = "AAPL"
if pred_reg.filter(F.col("ticker") == example_ticker).count() == 0:
    example_ticker = pred_reg.select("ticker").first()[0]

plot_reg_pdf = (
    pred_reg.filter(F.col("ticker") == example_ticker)
           .select("time", "close", "target_next_close", "pred_next_close", "ma7", "MA14")
           .orderBy("time")
           .toPandas()
)

plot_reg_pdf["time"] = pd.to_datetime(plot_reg_pdf["time"])

plt.figure(figsize=(14, 5))
plt.plot(plot_reg_pdf["time"], plot_reg_pdf["target_next_close"], label="Actual next close", linewidth=2)
plt.plot(plot_reg_pdf["time"], plot_reg_pdf["pred_next_close"], label="Predicted next close", linewidth=2)
plt.title(f"Close vs Prediction - {example_ticker}")
plt.xlabel("Time")
plt.ylabel("Price")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
plot_ma_pdf = (
    df_feat.filter(F.col("ticker") == example_ticker)
          .select("time", "close", "ma7", "MA14", "MA30")
          .orderBy("time")
          .toPandas()
)

plot_ma_pdf["time"] = pd.to_datetime(plot_ma_pdf["time"])

plt.figure(figsize=(14, 5))
plt.plot(plot_ma_pdf["time"], plot_ma_pdf["close"], label="Close", linewidth=2)
plt.plot(plot_ma_pdf["time"], plot_ma_pdf["ma7"], label="MA7", linewidth=1.5)
plt.plot(plot_ma_pdf["time"], plot_ma_pdf["MA14"], label="MA14", linewidth=1.5)
plt.plot(plot_ma_pdf["time"], plot_ma_pdf["MA30"], label="MA30", linewidth=1.5)
plt.title(f"Close vs MA - {example_ticker}")
plt.xlabel("Time")
plt.ylabel("Price")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Heatmap correlation
if not corr_pdf.empty:
    plt.figure(figsize=(8, 6))
    plt.imshow(corr_pdf.values, interpolation="nearest", aspect="auto")
    plt.colorbar(label="Correlation")
    plt.xticks(range(len(corr_pdf.columns)), corr_pdf.columns, rotation=45)
    plt.yticks(range(len(corr_pdf.index)), corr_pdf.index)
    plt.title("Correlation matrix of daily returns")
    plt.tight_layout()
    plt.show()

## 12. Ket luan

### Nhan xet thi truong
- Co su khac biet ve avg return/volatility giua nhom VN va US.
- Correlation giua cac ma cho thay mot so cap co phieu dong bien ro rang.

### Hieu qua mo hinh
- Linear Regression cung cap baseline cho du doan gia ngay tiep theo (RMSE).
- Logistic Regression du doan huong tang/giam (Accuracy, AUC).

### Han che
- Thi truong co tinh phi tuyen, phi dung, nhieu nhiu.
- Feature ky thuat co the chua du de bat duoc bien dong lon do tin tuc/vi mo.

### Goi y cai thien (Bonus)
- Thu mo hinh nang cao: XGBoost, LightGBM, LSTM/Transformer cho time series.
- Backtest theo rolling window walk-forward de danh gia sat thuc te hon.
- Bo sung du lieu ngoai sinh: lai suat, chi so thi truong, sentiment tin tuc.

### Vi sao khong duoc shuffle du lieu
- Shuffle pha vo quan he nhan qua theo thoi gian.
- Co the vo tinh dua thong tin tuong lai vao train, tao leakage va ket qua ao.

In [ ]:
# Tu chon: dung Spark khi ket thuc
# spark.stop()